# BrightPath Tutoring - Agent-Powered Business Assistant

This notebook demonstrates the business identity, separately labeled persona and instructions, three-way router, MCP-style tool schemas, Gemini interaction, short-term memory, tool execution, reflection, and Gradio interface required by the assignment.

## 1. Setup

Install dependencies once, then place `GEMINI_API_KEY` in a local `.env` file. Never paste the key into this notebook or commit `.env`.

In [ ]:
# Uncomment in a fresh Colab or Jupyter environment:
# %pip install -q -r requirements.txt

from brightpath_agent import (
    BrightPathAgent, INSTRUCTIONS, PERSONA, TOOL_SCHEMAS, classify_intent
)
print('Imports successful')

## 2. PERSONA - who the agent is

The persona controls identity, voice, and interaction style. It is deliberately separate from operational rules.

In [ ]:
print(PERSONA)

## 3. INSTRUCTIONS - operational guardrails

These rules prevent invented prices or schedules, control lead collection, require logging for feedback and unknown questions, and protect secrets.

In [ ]:
print(INSTRUCTIONS)

## 4. Router

Before generation, every incoming message is routed to `general_question`, `lead_collection`, or `feedback_unanswered`.

In [ ]:
router_examples = [
    'What does private university physics cost?',
    'I want to enroll in Python tutoring.',
    'I have a complaint about a missed lesson.',
    'Do you repair laptops?',
]
for example in router_examples:
    print(f'{classify_intent(example).value:24} | {example}')

## 5. MCP-style tool definitions

The schemas document each tool's name, description, parameters, required fields, and expected output. The callable implementations append JSON Lines records and print visible tool-call evidence.

In [ ]:
import json
print(json.dumps(TOOL_SCHEMAS, indent=2))

## 6. Live Gemini agent

The current `google-genai` SDK receives the system prompt, router result, recent history, and the relevant Python function. Automatic function calling executes the selected local tool and returns a user-facing response. The bonus reflection pass logs both the draft and revised reply.

In [ ]:
agent = BrightPathAgent(enable_reflection=True)
history = []
result = agent.answer('What English tutoring do you offer for university students?', history)
print('Route:', result.intent.value)
print('Reply:', result.reply)

## 7. Gradio ChatInterface

Run this cell to launch the demonstration chatbot.

In [ ]:
import gradio as gr

def respond(message, history):
    return agent.chat(message, history)

demo = gr.ChatInterface(
    fn=respond,
    title='BrightPath Tutoring Assistant',
    description='Ask about subjects, tutors, prices, schedules, recommendations, enrollment, or feedback.',
)
demo.launch()

## 8. Verification

From a terminal in this folder, run `pytest -q` for deterministic tests and `python scripts/smoke_test_live.py` for a real three-route Gemini test. Runtime records appear under `data/` and are ignored by Git.